In [ ]:
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random

In [ ]:
positives = []
negatives = []
import json
with open("data/sentiments_2.json", "r") as f:
    data = json.load(f)
for v in data:
    if v[1] == "pos": positives.append(v[0])
    if v[1] == "neg": negatives.append(v[0])

positives = positives[:200]
negatives = negatives[:100]

In [ ]:
entities = ["room"]

r = "{entity} {}"
r_ind = "{entity} is not {}"
r_amb = "{entity} is {a} but also {b}"

hotel_reviews_2 = []

for entity in entities:
    # Positives
    for p in positives:
        hotel_reviews_2.append({
            "entity": entity,
            "review": r.format(p, entity=entity),
            "sentiment": "positive"
        })

    for p in positives[:50]:
        hotel_reviews_2.append({
            "entity": entity,
            "review": r_ind.format(p, entity=entity),
            "sentiment": "indirect_negative"
        })

    # Negatives
    for n in negatives:
        hotel_reviews_2.append({
            "entity": entity,
            "review": r.format(n, entity=entity),
            "sentiment": "negative"
        })

    for n in negatives[:50]:
        hotel_reviews_2.append({
            "entity": entity,
            "review": r_ind.format(n, entity=entity),
            "sentiment": "indirect_positive"
        })

    # Ambiguous
    for p, n in zip(positives[:50], negatives[:50]):
        kp = random.choice(positives[:10])
        kn = random.choice(negatives[:10])

        hotel_reviews_2.append({
            "entity": entity,
            "review": r_amb.format(a=kp, b=kn, entity=entity),
            "sentiment": "ambiguous"
        })

        hotel_reviews_2.append({
            "entity": entity,
            "review": r_amb.format(a=n, b=kp, entity=entity),
            "sentiment": "ambiguous"
        })


In [4]:
sentences = [d["review"] for d in hotel_reviews_2]
sentiments = [d["sentiment"] for d in hotel_reviews_2]
entities_list = [d["entity"] for d in hotel_reviews_2]

In [5]:
sentiment_colors = {
    "negative": "#E69F00",           # Orange
    "positive": "#0072B2",           # Blue
    "indirect_positive": "#D3D3D3",  # Sky blue
    "indirect_negative": "#E69F00",  # Red-orange,
    "ambiguous": "#D3D3D3"
}

# Assign shapes to sentiment types for plotting
sentiment_shapes = {
    "positive": "o",          # Circle
    "negative": "s",          # Square
    "indirect_positive": "P", # Triangle up
    "indirect_negative": "X", # Triangle down,
    "ambiguous": "D"
}


In [ ]:
# -------------------------
from utils.SentiCSEmbeddings import SentiCSEmbeddings



# -------------------------
# Models
# -------------------------
models = {
    "all-mpnet-base-v2": SentenceTransformer("all-mpnet-base-v2", device="cpu"),
    "StanceSBERT": SentenceTransformer("vahidthegreat/StanceAware-SBERT", device="cpu"),
    "TourCSE": SentenceTransformer("data/sbert/hotel/all-mpnet-base-v2_lora_guided_Llama-3.1-8B-Instruct_v2", device="cpu"),
    "GTE-TourCSE": SentenceTransformer("data/sbert/hotel/gte-modernbert-base_lora_guided_Llama-3.1-8B-Instruct_v2", device="cpu"),
    "TourCSE_simcse": SentenceTransformer("data/sbert/export/hotel/all-mpnet-base-v2_lora_simcse", device="cpu"),
    "TourCSE_senticse": SentenceTransformer("data/sbert/export/hotel/all-mpnet-base-v2_lora_guided_senticse", device="cpu"),
    "GTE_modernbert": SentenceTransformer("Alibaba-NLP/gte-modernbert-base", device="cpu"),
    "SentiCSE": SentiCSEmbeddings("DILAB-HYU/SentiCSE"),
}

# -------------------------
# Embeddings + UMAP
# -------------------------

embeddings_dict = {}

for name, model in models.items():
    # Encode sentences
    emb = model.encode(sentences, show_progress_bar=True)
    embeddings_dict[name] = emb

No sentence-transformers model found with name vahidthegreat/StanceAware-SBERT. Creating a new one with mean pooling.
/home/mazais/Github/opinioncse-v3/.venv/lib/python3.12/site-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['qalora_group_size', 'target_parameters', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

100%|██████████| 500/500 [00:29<00:00, 16.72it/s]


In [7]:
from umap import UMAP
from sklearn.preprocessing import MinMaxScaler

In [8]:
def reduce_embeddings(n_neighbors, min_dist, emb):
    umap = UMAP(n_neighbors=n_neighbors, min_dist=min_dist, metric='cosine')
    emb_2d = umap.fit_transform(emb)
    scaler = MinMaxScaler()
    emb_2d_norm = scaler.fit_transform(emb_2d)
    return emb_2d_norm

In [9]:
def display_metric(umap_dict, save_fig):

    import numpy as np
    import pandas as pd
    from sklearn.metrics import (
        silhouette_score,
        calinski_harabasz_score,
        davies_bouldin_score,
    )

    # --------------------------------------------------
    # 1. Sentiment merging
    # --------------------------------------------------

    merge_labels = {
        "positive": ["positive"],
        "negative": ["negative", "indirect_negative"],
        "ambiguous": ["ambiguous", "indirect_positive"],
    }

    def map_sentiment(s):
        for merged_label, original_labels in merge_labels.items():
            if s in original_labels:
                return merged_label
        return None  # unknown → discard


    # Original sentiments: list[str]
    merged_sentiments = pd.Series([map_sentiment(s) for s in sentiments])

    # --------------------------------------------------
    # 2. Remove ambiguous AND unknown items (build mask)
    # --------------------------------------------------

    mask = merged_sentiments.notna() & (merged_sentiments != "ambiguous")

    print(f"Removed {(~mask).sum()} items out of {len(mask)}")
    print("Remaining label counts:")
    print(merged_sentiments[mask].value_counts())

    filtered_sentiments = merged_sentiments[mask].to_numpy()

    # --------------------------------------------------
    # 3. Encode labels (positive / negative)
    # --------------------------------------------------

    filtered_sentiments = merged_sentiments[mask]  # keep as pandas Series

    unique_labels = {lbl: i for i, lbl in enumerate(sorted(filtered_sentiments.unique()))}
    numeric_labels = filtered_sentiments.map(unique_labels).to_numpy()

    # Sanity checks
    assert set(unique_labels.keys()) == {"positive", "negative"}
    assert len(numeric_labels) == mask.sum()

    # --------------------------------------------------
    # 4. Compute clustering metrics
    # --------------------------------------------------

    results = []

    for model_name, emb in umap_dict.items():
        print(f"Computing metrics for: {model_name}")

        # Filter embeddings with the same mask
        emb_filtered = emb[mask.to_numpy()]

        # Safety check
        assert emb_filtered.shape[0] == len(numeric_labels)

        sil = silhouette_score(emb_filtered, numeric_labels)
        cal = calinski_harabasz_score(emb_filtered, numeric_labels)
        db = davies_bouldin_score(emb_filtered, numeric_labels)

        results.append({
            "model": model_name,
            "silhouette": sil,
            "calinski_harabasz": cal,
            "davies_bouldin": db,
        })

    df_metrics = pd.DataFrame(results)
    df_metrics["silhouette"] = df_metrics["silhouette"].round(2)
    df_metrics["davies_bouldin"] = df_metrics["davies_bouldin"].round(2)
    df_metrics["calinski_harabasz"] = df_metrics["calinski_harabasz"].round(0).astype(int)

    print(df_metrics)

    if save_fig:
        df_metrics.to_csv("data/article/appendix_sentiment/metric.csv")

In [10]:
def plot_main(umap_dict, save_fig=False):

    # Prepare figure: one row, one column per model
    n_models = len(umap_dict)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 6), constrained_layout=True)

    if n_models == 1:
        axes = [axes]  # Make iterable if only one model

    for ax, (model_name, emb_2d) in zip(axes, umap_dict.items()):
        
        for s in set(sentiments):
            # Indices of sentences for this sentiment
            idx = [i for i, t in enumerate(sentiments) if t == s]
            x = emb_2d[idx, 0]
            y = emb_2d[idx, 1]
            c = [sentiment_colors[sentiments[i]] for i in idx]
            
            
            ax.scatter(
                x, y,
                c=c,
                marker=sentiment_shapes.get(s, "o"),
                label=s,
                edgecolor="k",
                linewidths=1,
                s=120,
                alpha=0.8
            )
        

        ax.grid(True, linestyle="--", alpha=0.3)

    # Legend outside

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, [l.upper().replace("_", " ") for l in labels], loc="upper center", ncol=len(set(sentiments)), fontsize=18, frameon=False, bbox_to_anchor=(0.5, 1.1))

    if save_fig: fig.savefig("data/article/umap_sentiment.png", dpi=300, bbox_inches="tight")

    # Then display
    plt.show()

In [11]:
import matplotlib.patches as mpatches

def save_fig_appendix(umap_dict):

   for model_name, emb_2d in umap_dict.items():
    fig, ax = plt.subplots(figsize=(8, 8))  # <-- defines both fig and ax

    # Scatter plot per topic
    for s in set(sentiments):
        idx = [i for i, t in enumerate(sentiments) if t == s]
        x = emb_2d[idx, 0]
        y = emb_2d[idx, 1]
        c = [sentiment_colors[sentiments[i]] for i in idx]
        
        ax.scatter(
            x, y,
            c=c,
            marker=sentiment_shapes.get(s, "o"),
            label=s,
            edgecolor="k",
            linewidths=1,
            s=120,
            alpha=0.8
        )

        ax.grid(True, linestyle="--", alpha=0.3)
    
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=len(set(sentiments)),  # adjust if too wide
        fontsize=10,  # smaller font
        frameon=False,
        bbox_to_anchor=(0.5, 0.95)
    )

    # Save figure separately
    output_file = f"data/article/appendix_sentiment/{model_name}_umap_topics.png"
    plt.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {output_file}")

In [14]:
def interactive_plot(n_neighbors, min_dist, save_figures):
    print(n_neighbors, min_dist, save_figures)
    umap_dict = dict()
    for model in ['all-mpnet-base-v2', 'StanceSBERT', 'TourCSE']:
        umap_dict[model] = reduce_embeddings(n_neighbors, min_dist, embeddings_dict[model])
    
    plot_main(umap_dict, save_figures)

    if save_figures:
        for model in ['GTE-TourCSE', 'TourCSE_simcse', 'TourCSE_senticse', 'GTE_modernbert', 'SentiCSE']:
            umap_dict[model] = reduce_embeddings(n_neighbors, min_dist, embeddings_dict[model])
    
        save_fig_appendix(umap_dict)

    display_metric(umap_dict, save_figures)

In [15]:
import ipywidgets as widgets
from IPython.display import display

save_checkbox = widgets.Checkbox(
    value=False,
    description="Save Figures",
    indent=False
)

ui = widgets.VBox([
    widgets.IntSlider(
        min=5, max=200, step=5, value=50,
        description="Neighbors"
    ),
    widgets.FloatSlider(
        min=0.0, max=1, step=0.05, value=0.15,
        description="Min dist"
    ),
    save_checkbox
])

out = widgets.interactive_output(
    interactive_plot,
    {
        "n_neighbors": ui.children[0],
        "min_dist": ui.children[1],
        "save_figures": ui.children[2],  # pass checkbox value
    }
)

display(ui, out)


Output()